# 1 · `data_preprocess_pv.ipynb`

Process the raw PV power history: **interpolate** the 1-min log onto a 10-second grid, then
**filter** out gaps longer than 1 hour and any negative readings. Needed so images can be matched to
a PV value.

**Needs in Drive:** a raw `{Year}_pv_raw.csv` at `SKIPPD/`.
Source: https://purl.stanford.edu/sm043zf7254

> **Reconstruction note.** Clean, Colab-runnable reimplementation following the purpose the SKIPP'D
> README describes — not a byte-for-byte copy of the authors' notebook. **Image-only pipeline: no
> video processing.** All data lives in `/content/drive/MyDrive/SKIPPD/`.


## Mount Google Drive

Everything reads from and writes to `/content/drive/MyDrive/SKIPPD/`.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/SKIPPD"
os.makedirs(BASE, exist_ok=True)
print("Working folder:", BASE)
print("Contents:", os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/SKIPPD
Contents: ['2019_pv_raw.csv']


## Imports

In [6]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 20)

## Load raw PV CSV

Set the file name and adjust the column names to match your CSV.

In [12]:
PV_CSV    = os.path.join(BASE, "2019_pv_raw.csv")   # raw yearly PV file
TIME_COL  = "Date"             # timestamp column in the raw file
POWER_COL = "Huang_E4102_kW"   # PV power column in the raw file

pv = pd.read_csv(PV_CSV)
print(pv.head())
print("sample date value:", repr(pv[TIME_COL].iloc[0]))

# robust parse: unparseable rows -> NaT, then dropped
pv[TIME_COL] = pd.to_datetime(pv[TIME_COL], errors="coerce")
pv[POWER_COL] = pd.to_numeric(pv[POWER_COL], errors="coerce")
pv = (pv[[TIME_COL, POWER_COL]]
      .dropna()
      .sort_values(TIME_COL)
      .reset_index(drop=True))

print(f"Parsed {len(pv)} rows | range: {pv[TIME_COL].min()} .. {pv[TIME_COL].max()}")

                  Date  Huang_E4102_kW
0  2019-01-01T00:00:00       -0.084869
1  2019-01-01T00:01:00       -0.084093
2  2019-01-01T00:02:00       -0.084622
3  2019-01-01T00:03:00       -0.084654
4  2019-01-01T00:04:00       -0.084382
sample date value: '2019-01-01T00:00:00'
Parsed 431041 rows | range: 2019-01-01 00:00:00 .. 2019-10-27 09:00:00


## Step 1 — interpolate onto a 10-second grid

In [13]:
pv_idx = pv.set_index(TIME_COL)
grid   = pd.date_range(pv_idx.index.min(), pv_idx.index.max(), freq="10S")
pv_10s = (pv_idx.reindex(pv_idx.index.union(grid))
                .interpolate("time")
                .reindex(grid)
                .rename(columns={POWER_COL: "pv"}))
print("10s-grid samples:", len(pv_10s))
pv_10s.head()

/tmp/ipykernel_6580/3157136382.py:2: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  grid   = pd.date_range(pv_idx.index.min(), pv_idx.index.max(), freq="10S")


10s-grid samples: 2586601


,pv
2019-01-01 00:00:00,-0.084869
2019-01-01 00:00:10,-0.084740
2019-01-01 00:00:20,-0.084610
2019-01-01 00:00:30,-0.084481
2019-01-01 00:00:40,-0.084352


## Step 2 — filter invalid data

Drop points >1 h from any real record, or with negative PV.

In [14]:
real = pv_idx.index.values.astype("datetime64[ns]").astype("int64")
gp   = pv_10s.index.values.astype("datetime64[ns]").astype("int64")
j    = np.clip(np.searchsorted(real, gp), 1, len(real) - 1)
gap  = np.minimum(np.abs(gp - real[j - 1]), np.abs(gp - real[j])) / 1e9

valid = (gap <= 3600) & (pv_10s["pv"].values >= 0)
pv_clean = pv_10s[valid].copy()
print(f"Kept {valid.sum()} / {len(valid)} points")

Kept 1272323 / 2586601 points


## Save back to Drive

In [15]:
OUT_PV = os.path.join(BASE, "pv_processed_10s.csv")
pv_clean.to_csv(OUT_PV)
print("Saved ->", OUT_PV)
pv_clean.head()

Saved -> /content/drive/MyDrive/SKIPPD/pv_processed_10s.csv


,pv
2019-01-01 07:35:40,0.034080
2019-01-01 07:35:50,0.070947
2019-01-01 07:36:00,0.107814
2019-01-01 07:36:10,0.129527
2019-01-01 07:36:20,0.151240
